# ENDICEX · Atlas exploratorio de indicadores sociales y financieros

**Versión organizada en cuatro bloques | Muestra de 111 entidades, 101 con cuentas rendidas (2024).**

El objetivo es describir  **cada indicador** considerado en el artículo, tanto globalmente como por estrato poblacional. La población **no es un indicador analizado**: se usa únicamente para agrupar entidades. No se calculan correlaciones, PCA ni K-means.

### Índice

1. **Indicadores sociales · análisis individual** (111 entidades)
2. **Indicadores sociales · comparación por estratos** (111 entidades)
3. **Indicadores financieros · análisis individual** (101 entidades)
4. **Indicadores financieros · comparación por estratos** (101 entidades)
5. **Tablas, señalización de extremos, trazabilidad y descarga**

**Indicadores sociales:** renta media por persona, Gini, pobreza total (umbral del 60 %).  
**Indicadores financieros:** gasto municipal, gasto por habitante, endeudamiento por habitante e inversión por habitante.

**Estratos:** <5.000; 5.000–19.999; 20.000–99.999; 100.000 o más habitantes.

**Importante:** los ceros financieros de las 10 entidades sin cuentas rendidas no se interpretan como observaciones. Se distinguen las ocho imputaciones de `END HAB`, que no representan deuda directamente observada. La muestra no es representativa y los valores extremos no constituyen irregularidades.

**Entrada:** `ENDICEX_111_Anonimo_END_HAB_Imputado.xlsx` con las hojas `Muestra_111` y `Auditoria_END_HAB`. Si estás en Colab, sube el Excel al ejecutar la primera celda de código. El Excel original no se modifica.


In [ ]:
# ============================================================
# 0. CONFIGURACIÓN, CARGA Y VALIDACIÓN DE LA BASE ENDICEX
# ============================================================
# Inicialización de bibliotecas, parámetros gráficos y carpetas.
# Carga y validación de la muestra de 111 entidades locales.
#
# Indicadores sociales: 111 entidades.
# Indicadores financieros: 101 entidades con cuentas rendidas.
# Población: variable de estratificación, no de análisis.
# Estratos: <5.000 | 5.000–19.999 | 20.000–99.999 | ≥100.000.
#
# Control de las 8 imputaciones de endeudamiento por habitante.
# Los ceros financieros sin rendición se consideran datos ausentes.
# Se verifica la integridad y trazabilidad antes del análisis.
# ============================================================

from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
import warnings, math, re, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter
from IPython.display import display, Markdown, Image
from PIL import Image as PILImage, ImageDraw, ImageFont

warnings.filterwarnings('ignore', category=FutureWarning)
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams.update({'font.family':'DejaVu Sans','svg.fonttype':'none', 'axes.spines.right':False,
                     'axes.spines.top':False,'figure.dpi':110,'savefig.dpi':165})
ARCHIVO = 'ENDICEX_111_Anonimo_END_HAB_Imputado.xlsx'
HOJA = 'Muestra_111'
CARPETA = Path('ENDICEX_EDA_4_BLOQUES_SALIDAS')
FIG = CARPETA/'figuras'
TAB = CARPETA/'tablas'
for d in (CARPETA,FIG,TAB): d.mkdir(exist_ok=True,parents=True)
SOC = ['renta_media_persona','gini','pobreza_total_60']
FIN = ['GASTO MUNICIPAL','GTO HAB','END HAB','INV POR HAB']
VARIABLES = SOC + FIN
NOMBRES = {'renta_media_persona':'Renta media por persona (€)',
           'gini':'Índice de Gini','pobreza_total_60':'Pobreza total · umbral 60 % (%)',
           'GASTO MUNICIPAL':'Gasto municipal (€)','GTO HAB':'Gasto por habitante (€/hab.)',
           'END HAB':'Endeudamiento por habitante (€/hab.)',
           'INV POR HAB':'Inversión por habitante (€/hab.)'}
ESTRATOS = ['<5.000','5.000–19.999','20.000–99.999','100.000 o más']
PALETA = dict(zip(ESTRATOS,['#17789C','#449A83','#CC884A','#735F9B']))
AZUL = '#226F93'; NARANJA='#C57B47'; IMPUT='#DD9F24'; GRIS='#283848'

ruta = Path(ARCHIVO)
if not ruta.exists():
    try:
        from google.colab import files
        print('Sube el Excel imputado ENDICEX con las hojas Muestra_111 y Auditoria_END_HAB:')
        subida=files.upload()
        if ARCHIVO in subida: ruta=Path(ARCHIVO)
        else:
            candidatos=[Path(k) for k in subida if k.lower().endswith('.xlsx')]
            if len(candidatos)!=1: raise ValueError('Sube exactamente un archivo Excel .xlsx.')
            ruta=candidatos[0]
    except ImportError as exc:
        raise FileNotFoundError(f'Coloca {ARCHIVO} junto al notebook.') from exc
xls=pd.ExcelFile(ruta)
if not {'Muestra_111','Auditoria_END_HAB'}.issubset(xls.sheet_names):
    raise ValueError('Faltan las hojas Muestra_111 y/o Auditoria_END_HAB.')
df=pd.read_excel(ruta,sheet_name=HOJA,dtype={'codigo_ine':str,'id_aleatorio':str})
audit=pd.read_excel(ruta,sheet_name='Auditoria_END_HAB',dtype={'codigo_ine':str,'id_aleatorio':str})
df['codigo_ine']=df['codigo_ine'].astype('string').str.zfill(5)
audit['codigo_ine']=audit['codigo_ine'].astype('string').str.zfill(5)
required=['codigo_ine','id_aleatorio','poblacion','estrato','cuentas 24']+VARIABLES
missing=set(required)-set(df.columns)
if missing: raise ValueError('Columnas no encontradas: '+str(sorted(missing)))
for v in VARIABLES+['cuentas 24']:
    df[v]=pd.to_numeric(df[v],errors='coerce')
if len(df)!=111 or df['codigo_ine'].nunique()!=111: raise ValueError('Deben existir 111 códigos INE únicos.')
if int(df['cuentas 24'].eq(1).sum())!=101 or int(df['cuentas 24'].eq(0).sum())!=10:
    raise ValueError('La partición debe ser 101 con rendición y 10 sin rendición.')
if not set(df['estrato'].dropna()).issubset(set(ESTRATOS)):
    raise ValueError('Se encontró una denominación de estrato inesperada.')
if len(audit)!=8 or audit['codigo_ine'].nunique()!=8:
    raise ValueError('La auditoría END HAB debe contener exactamente 8 imputaciones.')
imp=set(audit['codigo_ine'].dropna())
if not imp.issubset(set(df['codigo_ine'])): raise ValueError('Identificadores imputados no coinciden con muestra.')
df['end_hab_imputado']=df['codigo_ine'].isin(imp)
if not df.loc[df.end_hab_imputado,'cuentas 24'].eq(1).all():
    raise ValueError('Hay END HAB imputado en una entidad sin rendición 2024.')
# Ceros financieros de entidades sin cuentas: ausencia estructural; NO valores reales.
df.loc[df['cuentas 24'].eq(0),FIN]=np.nan
social=df.copy()
financiero=df.loc[df['cuentas 24'].eq(1)].copy()
if not all(financiero[v].notna().sum()==101 for v in FIN):
    print('Aviso: alguna variable financiera tiene ausencias entre las 101 entidades con cuentas.')
print('BASE VALIDADA: 111 entidades; 101 con rendición; 10 sin rendición; 8 valores END HAB imputados.')
print('Entidades por estrato:',df['estrato'].value_counts().reindex(ESTRATOS).fillna(0).astype(int).to_dict())
display(df[['id_aleatorio','estrato','cuentas 24']+SOC].head(3))


## Funciones de visualización y diagnóstico
Cada gráfico mantiene visible el universo empleado y evita usar población como eje sustantivo.

In [ ]:
# ============================================================
# 1. FUNCIONES AUXILIARES: ESTADÍSTICA DESCRIPTIVA Y GRÁFICOS
# ============================================================
# Definición de funciones reutilizables para el análisis EDA.
#
# Cálculo de estadísticos descriptivos: media, mediana,
# desviación típica, cuartiles, IQR, asimetría y valores atípicos.
#
# Generación de fichas individuales para cada indicador:
# histogramas, boxplots, ECDF y nubes de puntos ordenadas.
#
# Generación de fichas comparativas por estrato poblacional:
# boxplots, violin plots, ECDF y medianas con rangos IQR.
#
# Identificación visual de los 8 valores imputados de END HAB.
# Aplicación de escala logarítmica al gasto municipal.
#
# Exportación automática de figuras en formatos PNG y SVG.
# Análisis exclusivamente descriptivo, sin inferencia causal.
# ============================================================


def guardar(fig, nombre):
    fig.savefig(FIG/(nombre+'.png'), bbox_inches='tight',facecolor='white')
    fig.savefig(FIG/(nombre+'.svg'), bbox_inches='tight',facecolor='white')
    plt.close(fig)

def descriptivos(datos, variable, estrato=None):
    if estrato is not None: datos=datos.loc[datos.estrato.eq(estrato)]
    x=pd.to_numeric(datos[variable],errors='coerce').dropna().astype(float)
    if x.empty:
        return dict(variable=variable,estrato=estrato or 'TOTAL',n=0,ausentes=int(len(datos)),
                    media=np.nan,mediana=np.nan,desv_tipica=np.nan,p25=np.nan,p75=np.nan,
                    IQR=np.nan,min=np.nan,max=np.nan,asimetria=np.nan,atipicos_IQR=0)
    q1,q3=x.quantile([.25,.75]);iqr=q3-q1
    return dict(variable=variable,estrato=estrato or 'TOTAL',n=int(len(x)),
        ausentes=int(datos[variable].isna().sum()),media=x.mean(),mediana=x.median(),
        desv_tipica=x.std(),p25=q1,p75=q3,IQR=iqr,min=x.min(),max=x.max(),
        asimetria=x.skew(),atipicos_IQR=int(((x<q1-1.5*iqr)|(x>q3+1.5*iqr)).sum()))

def slug(variable):
    return re.sub('[^a-z0-9]+','_',variable.lower()).strip('_')

def log_gasto(variable, ax, eje):
    # El rango de gasto municipal supera tres órdenes de magnitud.
    if variable=='GASTO MUNICIPAL':
        if eje=='x': ax.set_xscale('log')
        else: ax.set_yscale('log')
        ax.grid(True,which='both',alpha=.22)

def aplicar_titulo(fig,bloque,variable,n,subtitulo):
    fig.suptitle(f'{bloque} · {NOMBRES[variable]}', fontsize=15,weight='bold',y=1.078)
    fig.text(.5,1.036,f'n={n} entidades · {subtitulo}',ha='center',va='bottom',fontsize=10,color='#526477')

def ficha_individual(datos,variable,numero):
    sub=datos.loc[datos[variable].notna()].copy().sort_values(variable)
    x=sub[variable].astype(float);color=AZUL if variable in SOC else NARANJA
    fig,axs=plt.subplots(2,2,figsize=(14,9),constrained_layout=True)
    ax=axs[0,0]
    if variable=='GASTO MUNICIPAL':
        bins=np.geomspace(max(x.min()*.98,1),x.max()*1.02,26)
    else:
        bins='fd'
    sns.histplot(x=x,bins=bins,color=color,edgecolor='white',ax=ax)
    ax.axvline(x.median(),color='#9B3D44',ls='--',lw=2,label='Mediana')
    ax.axvline(x.mean(),color=GRIS,ls=':',lw=2,label='Media')
    log_gasto(variable,ax,'x')
    ax.set_title('A · Histograma');ax.set_xlabel(NOMBRES[variable]);ax.legend(frameon=False)
    ax=axs[0,1]
    sns.boxplot(x=x,color=color,showmeans=True,ax=ax,width=.38,
                meanprops={'marker':'D','markerfacecolor':'white','markeredgecolor':'black'})
    sns.stripplot(x=x,color=GRIS,alpha=.40,size=3.4,jitter=.12,ax=ax)
    if variable=='END HAB':
        xi=sub.loc[sub.end_hab_imputado,variable]
        ax.scatter(xi,np.zeros(len(xi)),marker='^',s=96,color=IMPUT,edgecolor='#744D15',zorder=8,
                   label='Imputado (n=8)')
        ax.legend(frameon=False,loc='upper right')
    log_gasto(variable,ax,'x')
    ax.set_title('B · Boxplot y puntos observados');ax.set_xlabel(NOMBRES[variable]);ax.set_yticks([])
    ax=axs[1,0]
    sns.ecdfplot(x=x,color=color,lw=2.3,ax=ax)
    ax.axhline(.5,color='#9B3D44',ls='--',alpha=.6)
    log_gasto(variable,ax,'x')
    ax.set_title('C · Distribución acumulada (ECDF)');ax.set_xlabel(NOMBRES[variable]);ax.set_ylabel('Proporción acumulada')
    ax=axs[1,1]
    ranks=np.arange(1,len(sub)+1)
    ax.scatter(ranks,sub[variable],color=color,s=24,alpha=.75)
    if variable=='END HAB':
        ids=np.flatnonzero(sub.end_hab_imputado.values)
        ax.scatter(ranks[ids],sub.iloc[ids][variable],color=IMPUT,marker='^',s=76,
                   edgecolor='#744D15',label='Imputado',zorder=8)
        ax.legend(frameon=False)
    log_gasto(variable,ax,'y')
    ax.set_title('D · Nube de puntos ordenada');ax.set_xlabel('Orden de menor a mayor valor');ax.set_ylabel(NOMBRES[variable])
    aplicar_titulo(fig,'INDIVIDUAL',variable,len(sub),
        'la nube de puntos usa rango de observación, no población')
    guardar(fig,f'{numero}_individual_{slug(variable)}')

def ficha_estratos(datos,variable,numero):
    sub=datos.loc[datos[variable].notna()].copy()
    sub['estrato']=pd.Categorical(sub['estrato'],categories=ESTRATOS,ordered=True)
    color=AZUL if variable in SOC else NARANJA
    orden=ESTRATOS
    counts=sub['estrato'].value_counts().reindex(ESTRATOS).fillna(0).astype(int)
    etiquetas=[f'{s}\n(n={counts[s]})' for s in orden]
    fig,axs=plt.subplots(2,2,figsize=(15.5,10),constrained_layout=True)
    ax=axs[0,0]
    sns.boxplot(data=sub,x='estrato',y=variable,order=orden,color=color,ax=ax,width=.55,
                showfliers=False)
    sns.stripplot(data=sub,x='estrato',y=variable,order=orden,ax=ax,color=GRIS,
                  alpha=.55,size=3.4,jitter=.20)
    if variable=='END HAB':
        imp_df=sub.loc[sub.end_hab_imputado]
        for i,s in enumerate(orden):
            vals=imp_df.loc[imp_df.estrato.eq(s),variable]
            ax.scatter(np.full(len(vals),i),vals,s=95,marker='^',c=IMPUT,
                       edgecolors='#744D15',zorder=9)
        ax.text(.98,.96,'▲ valor imputado',transform=ax.transAxes,ha='right',va='top',color='#845C1A',fontsize=9)
    log_gasto(variable,ax,'y');ax.set_xticks(range(4),etiquetas);ax.tick_params(axis='x',labelsize=9)
    ax.set_title('A · Boxplot por estrato + puntos');ax.set_xlabel('Estrato de población');ax.set_ylabel(NOMBRES[variable])
    ax=axs[0,1]
    sns.violinplot(data=sub,x='estrato',y=variable,order=orden,ax=ax,
                   color=color,inner='quart',cut=0,linewidth=1)
    log_gasto(variable,ax,'y');ax.set_xticks(range(4),etiquetas);ax.tick_params(axis='x',labelsize=9)
    ax.set_title('B · Densidad tipo violín (orientativa con n pequeño)')
    ax.set_xlabel('Estrato de población');ax.set_ylabel(NOMBRES[variable])
    ax=axs[1,0]
    for s in orden:
        valores=sub.loc[sub.estrato.eq(s),variable].dropna()
        if len(valores): sns.ecdfplot(x=valores,ax=ax,label=s,color=PALETA[s],lw=2.2)
    log_gasto(variable,ax,'x')
    ax.set_title('C · ECDF comparada por estrato');ax.set_xlabel(NOMBRES[variable]);ax.set_ylabel('Proporción acumulada')
    ax.legend(title='Estrato',fontsize=8,title_fontsize=9,frameon=False)
    ax=axs[1,1]
    for i,s in enumerate(orden):
        v=sub.loc[sub.estrato.eq(s),variable].dropna()
        if v.empty: continue
        q1,q3=np.quantile(v,[.25,.75]);med=float(np.median(v))
        ax.plot([q1,q3],[i,i],color=PALETA[s],lw=8,alpha=.72,solid_capstyle='round')
        ax.scatter(med,i,color=GRIS,s=75,zorder=5,marker='D',edgecolor='white')
    ax.set_yticks(range(4),etiquetas);ax.tick_params(axis='y',labelsize=9);ax.invert_yaxis()
    log_gasto(variable,ax,'x')
    ax.set_title('D · Mediana (rombo) y rango intercuartílico');ax.set_xlabel(NOMBRES[variable]);ax.set_ylabel('')
    aplicar_titulo(fig,'POR ESTRATOS',variable,len(sub),
        'comparación descriptiva; no representa inferencia estadística')
    guardar(fig,f'{numero}_estratos_{slug(variable)}')

print('Funciones de visualización preparadas.')


## Bloque 1 · Indicadores sociales: análisis individual (111 entidades)
Histograma, boxplot, ECDF y nube de puntos ordenada por cada variable.

In [ ]:
# ============================================================
# BLOQUE 1. ANÁLISIS INDIVIDUAL DE LOS INDICADORES SOCIALES
# ============================================================
# Análisis exploratorio univariante de las 111 entidades.
#
# Indicadores:
#   1. Renta media por persona (€).
#   2. Índice de Gini (desigualdad).
#   3. Pobreza total bajo el umbral del 60 %.
#
# Para cada indicador se generan cuatro visualizaciones:
#   A. Histograma con media y mediana.
#   B. Boxplot con observaciones individuales.
#   C. Distribución acumulada empírica (ECDF).
#   D. Nube de puntos ordenada por valor.
#
# Objetivo: caracterizar distribuciones, dispersión, asimetría
# y valores extremos, sin considerar la población como indicador.
#
# Exportación de tres fichas en formatos PNG y SVG.
# Previsualización de la ficha de renta media por persona.
# ============================================================

for i,v in enumerate(SOC,1):
    ficha_individual(social,v,'01_%02d'%i)
print('Bloque 1 generado:',len(SOC),'fichas PNG + SVG.')
display(Image(filename=str(FIG/'01_01_individual_renta_media_persona.png'),width=800))


## Bloque 2 · Indicadores sociales por estratos (111 entidades)
Comparación de las distribuciones en los cuatro estratos.

In [ ]:
# ============================================================
# BLOQUE 2. INDICADORES SOCIALES POR ESTRATOS DE POBLACIÓN
# ============================================================
# Comparación descriptiva de los indicadores sociales entre
# los cuatro estratos poblacionales de la muestra ENDICEX.
#
# Indicadores:
#   1. Renta media por persona (€).
#   2. Índice de Gini.
#   3. Pobreza total bajo el umbral del 60 %.
#
# Estratos:
#   - Menos de 5.000 habitantes.
#   - 5.000 a 19.999 habitantes.
#   - 20.000 a 99.999 habitantes.
#   - 100.000 habitantes o más.
#
# Para cada indicador se generan cuatro visualizaciones:
#   A. Boxplot por estrato con puntos observados.
#   B. Gráfico de violín con cuartiles.
#   C. ECDF comparada entre estratos.
#   D. Mediana e intervalo intercuartílico por estrato.
#
# Objetivo: comprobar diferencias de nivel, dispersión y forma
# de la distribución entre grupos de tamaño poblacional.
#
# Enfoque estrictamente descriptivo, sin inferencia estadística.
# Exportación de tres fichas en formatos PNG y SVG.
# Previsualización de la ficha de pobreza total.
# ============================================================

# BLOQUE 2 · Tres indicadores sociales | comparación entre cuatro estratos (n=111)
for i,v in enumerate(SOC,1):
    ficha_estratos(social,v,'02_%02d'%i)
print('Bloque 2 generado:',len(SOC),'fichas PNG + SVG.')
display(Image(filename=str(FIG/'02_03_estratos_pobreza_total_60.png'),width=800))


## Bloque 3 · Indicadores financieros: análisis individual (101 entidades)
Se excluyen las diez entidades sin rendición; `END HAB` distingue ocho valores imputados.

In [ ]:
# ============================================================
# BLOQUE 3. ANÁLISIS INDIVIDUAL DE LOS INDICADORES FINANCIEROS
# ============================================================
# Análisis exploratorio univariante de las 101 entidades con
# cuentas rendidas en 2024.
#
# Indicadores:
#   1. Gasto municipal (€).
#   2. Gasto por habitante (€/hab.).
#   3. Endeudamiento por habitante (€/hab.).
#   4. Inversión por habitante (€/hab.).
#
# Para cada indicador se generan cuatro visualizaciones:
#   A. Histograma con media y mediana.
#   B. Boxplot con observaciones individuales.
#   C. Distribución acumulada empírica (ECDF).
#   D. Nube de puntos ordenada por valor.
#
# En gasto municipal se aplica escala logarítmica cuando resulta
# necesario por su amplitud de rango.
#
# En endeudamiento por habitante se señalan visualmente los
# 8 valores imputados para mantener su trazabilidad.
#
# Objetivo: describir la distribución, heterogeneidad y valores
# extremos de los indicadores financieros válidos.
#
# Exportación de cuatro fichas en formatos PNG y SVG.
# Previsualización de la ficha individual de END HAB.
# ============================================================

# BLOQUE 3 · Cuatro indicadores financieros | fichas individuales (n=101)
for i,v in enumerate(FIN,1):
    ficha_individual(financiero,v,'03_%02d'%i)
print('Bloque 3 generado:',len(FIN),'fichas PNG + SVG.')
display(Image(filename=str(FIG/'03_03_individual_end_hab.png'),width=800))


## Bloque 4 · Indicadores financieros por estratos (101 entidades)
Los extremos IQR se identifican respecto al estrato de cada entidad.

In [ ]:
# ============================================================
# BLOQUE 4. INDICADORES FINANCIEROS POR ESTRATOS DE POBLACIÓN
# ============================================================
# Comparación descriptiva de los indicadores financieros entre
# los cuatro estratos poblacionales, considerando únicamente
# las 101 entidades con cuentas rendidas en 2024.
#
# Indicadores:
#   1. Gasto municipal (€).
#   2. Gasto por habitante (€/hab.).
#   3. Endeudamiento por habitante (€/hab.).
#   4. Inversión por habitante (€/hab.).
#
# Estratos:
#   - Menos de 5.000 habitantes.
#   - 5.000 a 19.999 habitantes.
#   - 20.000 a 99.999 habitantes.
#   - 100.000 habitantes o más.
#
# Para cada indicador se generan cuatro visualizaciones:
#   A. Boxplot por estrato con puntos observados.
#   B. Gráfico de violín con cuartiles.
#   C. ECDF comparada entre estratos.
#   D. Mediana e intervalo intercuartílico por estrato.
#
# En gasto municipal puede aplicarse escala logarítmica por la
# amplitud del rango. En END HAB se señalan los 8 valores
# imputados para preservar su trazabilidad analítica.
#
# Objetivo: identificar diferencias descriptivas entre estratos
# en nivel, dispersión y heterogeneidad financiera.
#
# Enfoque estrictamente descriptivo, sin inferencia estadística.
# Exportación de cuatro fichas en formatos PNG y SVG.
# Previsualización de la ficha estratificada de END HAB.
# ============================================================

# BLOQUE 4 · Cuatro indicadores financieros | comparación por estratos (n=101)
for i,v in enumerate(FIN,1):
    ficha_estratos(financiero,v,'04_%02d'%i)
print('Bloque 4 generado:',len(FIN),'fichas PNG + SVG.')
display(Image(filename=str(FIG/'04_03_estratos_end_hab.png'),width=800))


## Síntesis, trazabilidad y exportación
CSV descriptivos globales, por estrato, extremos IQR intraestrato y auditoría de las imputaciones.

In [ ]:
# ============================================================
# 5. TABLAS DESCRIPTIVAS, VALORES EXTREMOS Y EXPORTACIÓN
# ============================================================
# Consolidación de los resultados exploratorios de ENDICEX.
#
# Cálculo de estadísticos descriptivos globales y por estrato
# para los 3 indicadores sociales y los 4 financieros.
#
# Identificación de valores atípicos mediante el criterio IQR
# (1,5 veces el rango intercuartílico), aplicado por estrato.
#
# Registro de los valores extremos, sus límites estadísticos
# y la identificación seudonimizada de cada entidad.
#
# Incorporación de la trazabilidad de las 8 imputaciones
# de endeudamiento por habitante (END HAB).
#
# Exportación de cinco tablas CSV:
#   1. Estadísticos descriptivos globales.
#   2. Estadísticos descriptivos por estrato.
#   3. Valores atípicos identificados mediante IQR.
#   4. Auditoría de las imputaciones de END HAB.
#   5. Distribución de entidades por estrato y rendición.
#
# Generación de un archivo LEEME con la metodología,
# estructura del análisis y cautelas de interpretación.
#
# Creación de una previsualización gráfica con cuatro fichas
# representativas de los bloques sociales y financieros.
#
# Objetivo: garantizar la trazabilidad, reproducibilidad
# y documentación de los resultados del análisis exploratorio.
#
# Los valores atípicos constituyen señales descriptivas,
# no evidencias de irregularidades ni hallazgos de fiscalización.
# ============================================================

# Asegurar que todas las figuras necesarias existan antes de generar la portada
for i, v in enumerate(SOC, 1):
    p = FIG/f'01_%02d_individual_{slug(v)}.png'
    if not p.exists():
        ficha_individual(social, v, '01_%02d'%i)
    p2 = FIG/f'02_%02d_estratos_{slug(v)}.png'
    if not p2.exists():
        ficha_estratos(social, v, '02_%02d'%i)

for i, v in enumerate(FIN, 1):
    p = FIG/f'03_%02d_individual_{slug(v)}.png'
    if not p.exists():
        ficha_individual(financiero, v, '03_%02d'%i)
    p2 = FIG/f'04_%02d_estratos_{slug(v)}.png'
    if not p2.exists():
        ficha_estratos(financiero, v, '04_%02d'%i)

# 5 · Tablas descriptivas, extremos intraestrato, trazabilidad y empaquetado
res_total=[]
res_estrato=[]
outs=[]
for v in VARIABLES:
    data=social if v in SOC else financiero
    total=descriptivos(data,v)
    total['bloque']='Social' if v in SOC else 'Financiero'
    total['etiqueta']=NOMBRES[v]
    res_total.append(total)
    for estrato in ESTRATOS:
        d=descriptivos(data,v,estrato)
        d['bloque']='Social' if v in SOC else 'Financiero'
        d['etiqueta']=NOMBRES[v]
        res_estrato.append(d)
        sub=data.loc[data['estrato'].eq(estrato),['id_aleatorio','codigo_ine','end_hab_imputado',v]].dropna(subset=[v]).copy()
        if sub.empty: continue
        q1,q3=np.quantile(sub[v],[.25,.75]);iqr=q3-q1
        li,ls=q1-1.5*iqr,q3+1.5*iqr
        for _,r in sub.loc[(sub[v]<li)|(sub[v]>ls)].iterrows():
            outs.append({'bloque':d['bloque'],'variable':v,'estrato':estrato,
                         'id_aleatorio':r['id_aleatorio'],'valor':r[v],
                         'lim_inf_IQR':li,'lim_sup_IQR':ls,
                         'end_hab_imputado':bool(r['end_hab_imputado'])})

globales=pd.DataFrame(res_total)
por_estrato=pd.DataFrame(res_estrato)
valores_extremos=pd.DataFrame(outs,columns=['bloque','variable','estrato','id_aleatorio','valor','lim_inf_IQR','lim_sup_IQR','end_hab_imputado'])
for nombre,tabla in [('01_descriptivos_globales.csv',globales),
                     ('02_descriptivos_por_estrato.csv',por_estrato),
                     ('03_atipicos_IQR_por_estrato.csv',valores_extremos),
                     ('04_trazabilidad_END_HAB.csv',audit),
                     ('05_recuentos_estratos.csv',df.groupby(['estrato','cuentas 24'],observed=True).size().rename('n').reset_index())]:
    tabla.to_csv(TAB/nombre,index=False,sep=';',decimal=',',encoding='utf-8-sig')

lectura=f"""# ENDICEX · Atlas exploratorio · 4 bloques\n\n
## Índice\n1. Indicadores sociales individuales · 3 fichas (111 entidades).\n2. Indicadores sociales por estratos · 3 fichas (111 entidades).\n3. Indicadores financieros individuales · 4 fichas (101 entidades).\n4. Indicadores financieros por estratos · 4 fichas (101 entidades).\n\n
**Fuente:** `{ruta.name}`, hojas `Muestra_111` y `Auditoria_END_HAB`.\n\n
**Estratos:** {', '.join(ESTRATOS)}.\n
**Variables:** sociales {', '.join(SOC)}; financieras {', '.join(FIN)}.\n
**Población:** solo clasificación por estratos, nunca indicador sustantivo.\n
**Trazabilidad:** 111 entidades; 101 con rendición 2024; 10 sin rendición. Ocho END HAB imputados mediante medianas; se marcan con triángulo en gráficos pertinentes. Los valores imputados no son observados.\n
**Salida:** 14 fichas de 4 paneles en PNG y SVG; 5 tablas CSV de descriptivos, extremos por estrato y auditoría. Se excluye el Excel de entrada para evitar duplicar datos; súbelo al ejecutar el notebook.\n
**Interpretación:** muestra exploratoria no representativa. No hay pruebas de correlación, inferencia causal, PCA ni K-means. Los extremos IQR solo son indicios para revisión estadística, no hallazgos de fiscalización.\n
**Gasto municipal:** se representa en eje logarítmico positivo debido a la gran amplitud de su distribución.\n
**Gráficos individuales:** histograma, boxplot con observaciones, ECDF y nube de puntos ordenada (no se usa la población en el eje).\n
**Gráficos por estrato:** boxplot con puntos, violin orientativo, ECDF por estrato y mediana con IQR.\n"""
(CARPETA/'LEEME.md').write_text(lectura,encoding='utf-8')

# Portada visual del ZIP: miniaturas correspondientes a los cuatro bloques.
seleccion=[
  ('01 SOCIALES · INDIVIDUAL',FIG/'01_01_individual_renta_media_persona.png'),
  ('02 SOCIALES · ESTRATOS',FIG/'02_03_estratos_pobreza_total_60.png'),
  ('03 FINANCIEROS · INDIVIDUAL',FIG/'03_03_individual_end_hab.png'),
  ('04 FINANCIEROS · ESTRATOS',FIG/'04_03_estratos_end_hab.png')]
W,H=1900,1370
lienzo=PILImage.new('RGB',(W,H),'#F5F8FB')
draw=ImageDraw.Draw(lienzo)
try:
    font=ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf',24)
    large=ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf',35)
except OSError:
    font=ImageFont.load_default();large=font
draw.text((54,20),'ENDICEX  |  ATLAS EXPLORATORIO · 4 BLOQUES',fill='#183A50',font=large)
draw.text((56,66),'3 indicadores sociales (n=111) · 4 financieros (n=101) · población = estratificación',fill='#49677B',font=font)
for k,(titulo,p) in enumerate(seleccion):
    x=50+(k%2)*940; y=132+(k//2)*610
    draw.rounded_rectangle((x-10,y-8,x+906,y+585),radius=17,fill='white',outline='#D6E3EB',width=2)
    draw.text((x+20,y+5),titulo,fill='#174A68',font=font)
    im=PILImage.open(p).convert('RGB')
    im.thumbnail((866,510))
    lienzo.paste(im,(x+20+(866-im.width)//2,y+58))
portada=CARPETA/'PREVISUALIZACION_4_BLOQUES.png'
lienzo.save(portada)

display(globales[['etiqueta','n','mediana','IQR','atipicos_IQR']].round(2))
print('Figuras:',len(list(FIG.glob('*.png'))),'PNG y',len(list(FIG.glob('*.svg'))),'SVG')
print('Tabla de extremos: ',len(valores_extremos),'observaciones variable×estrato señaladas (pueden repetirse).')
display(Image(filename=str(portada),width=1000))

In [ ]:
# ============================================================
# 6. EMPAQUETADO FINAL Y DESCARGA DEL ATLAS EXPLORATORIO
# ============================================================
# Generación del archivo ZIP final con los resultados del
# análisis exploratorio de ENDICEX, organizado en cuatro bloques.
#
# Contenido del archivo:
#   1. Notebook Jupyter reproducible (.ipynb).
#   2. Catorce fichas gráficas en formato PNG.
#   3. Catorce fichas gráficas en formato SVG.
#   4. Cinco tablas de resultados en formato CSV.
#   5. Previsualización de los cuatro bloques analíticos.
#   6. Documento LEEME con metodología y trazabilidad.
#
# Compresión de los archivos conservando la estructura
# de carpetas y los nombres originales.
#
# Descarga automática del ZIP cuando se ejecuta en Google Colab.
# En otros entornos, el archivo permanece en el directorio local.
#
# Objetivo: facilitar la conservación, consulta, difusión
# y reproducibilidad del análisis exploratorio de ENDICEX.
#
# El Excel original no se incorpora al ZIP para evitar
# duplicidades y mantener separadas las fuentes y los resultados.
# ============================================================

# 6 · ZIP final y descarga (solo si ejecutas Google Colab)
notebook_origen=Path('ENDICEX_EDA_4_Bloques_Colab.ipynb')
zip_path=Path('ENDICEX_Atlas_EDA_4_Bloques.zip')
with ZipFile(zip_path,'w',ZIP_DEFLATED,compresslevel=8) as z:
    for p in sorted(CARPETA.rglob('*')):
        if p.is_file(): z.write(p,p.as_posix())
    if notebook_origen.exists(): z.write(notebook_origen,notebook_origen.name)
print('ZIP generado:',zip_path.resolve(),f'({zip_path.stat().st_size/1e6:.2f} MB)')
try:
    from google.colab import files
    files.download(str(zip_path))
except ImportError:
    print('Fuera de Colab, abre el ZIP en la carpeta de trabajo.')


---
**Cautela de interpretación:** el análisis es exclusivamente descriptivo, con un diseño de muestra exploratoria. No permite afirmar causalidad, representatividad ni irregularidades de fiscalización.